In [1]:
TOLERANCE = 1e-9
from typing import Dict, List, Tuple
import time
import numpy as np
from classHMM import HMM

In [ ]:
def forward(hmm, observations: str) -> Tuple[np.ndarray, float]:
    """Finds the total probability of the observed sequence across all paths given the HMM probabilities.

    Args:
        observations (str): observed symbols, i.e. "GGCACTGAA"

    Returns:
        Tuple[np.ndarray, float]: the forward trellis and the total log-probability

    Examples:
        >>> forward(hmm, 'G')
        (array([[-2.12026354],
                [-1.83258146]]), np.float64(-1.2729656758128873))
        >>> forward(hmm, 'GG')
        (array([[-2.12026354, -3.54737989],
                [-1.83258146, -2.91139113]]), np.float64(-2.486507931154974))
    """

    #Initialize matrix
    hmm.check_observations(observations)
    num_positions = len(observations)
    score_matrix = np.full((len(hmm.states), num_positions), -np.inf)
    #Get initial probability
    first_symbol = observations[0]

    for state in hmm.states:
         score_matrix[hmm.state_index[state], 0] = hmm.log_init[state] + hmm.log_emit[state][first_symbol]

    #Looping through observations and summing up scores
    for column in range(1, num_positions):
        symbol = observations[column]
        for current_state in hmm.states:
            incoming = []
            for previous_state in hmm.states:
                incoming.append(score_matrix[hmm.state_index[previous_state], column - 1]
                + hmm.log_trans[previous_state][current_state])
            score_matrix[hmm.state_index[current_state], column] = np.logaddexp.reduce(incoming) + hmm.log_emit[current_state][symbol]

    last_col_sum = np.logaddexp.reduce(score_matrix[:, -1])
    return score_matrix, last_col_sum


def backward(hmm, observations: str) -> Tuple[np.ndarray, float]:
    """Finds the total probability of the observed sequence across all paths given the HMM probabilities.

    Args:
        observations (str): observed symbols, i.e. "GGCACTGAA"

    Returns:
        Tuple[np.ndarray, float]: the backward trellis and the total log-probability

    Examples:
        >>> backward(hmm, 'G')
        (array([[0.],
                [0.]]), np.float64(-1.2729656758128873))
        >>> backward(hmm, 'GG')
        (array([[-1.42711636, 0.],
                [-1.07880966, 0.]]), np.float64(-2.486507931154974))
    """
    #Initialize Matrix
    hmm.check_observations(observations)
    num_positions = len(observations)
    score_matrix = np.full((len(hmm.states), num_positions), -np.inf)
    score_matrix[:,-1] = 0

    for column in range(num_positions - 2, -1, -1):
        next_symbol = observations[column + 1]
        for current_state in hmm.states:
            outgoing = []
            for next_state in hmm.states:
                outgoing.append(hmm.log_trans[current_state][next_state] + hmm.log_emit[next_state][next_symbol] + score_matrix[hmm
                                .state_index[next_state], column + 1])
            score_matrix[hmm.state_index[current_state], column] = np.logaddexp.reduce(outgoing)

    first_symbol = observations[0]
    finish_terms = []
    for state in hmm.states:
        finish_terms.append(hmm.log_init[state] + hmm.log_emit[state][first_symbol] + score_matrix[hmm.state_index[state], 0])
    log_prob = np.logaddexp.reduce(finish_terms)

    return score_matrix, log_prob


def forward_backward(hmm, observations: str) -> Dict[str, object]:
    """Calculates the posterior probability of each hidden state at each position given the entire observation sequence.

    Args:
        observations (str): observed symbols, i.e. "GGCACTGAA"

    Returns:
        Dict[str, object]: a dictionary containing:
            posterior: posterior probability matrix
            forward: forward log-probability matrix
            backward: backward log-probability matrix
            log_prob: total sequence log-probability
            
    Examples:
        >>> forward_backward(hmm, 'G')
        {'posterior': array([[0.42857143],
                            [0.57142857]]),
        'forward': array([[-2.12026354],
                        [-1.83258146]]),
        'backward': array([[0.],
                            [0.]]),
        'log_prob': np.float64(-1.2729656758128873)}
        >>> forward_backward(hmm, 'GG')
        {'posterior': array([[0.34615385, 0.34615385],
                            [0.65384615, 0.65384615]]),
        'forward': array([[-2.12026354, -3.54737989],
                        [-1.83258146, -2.91139113]]),
        'backward': array([[-1.42711636, 0.],
                            [-1.07880966, 0.]]),
        'log_prob': np.float64(-2.486507931154974)}
    """
    forward_matrix, forward_log_prob = forward(hmm, observations)
    backward_matrix, backward_log_prob = backward(hmm, observations)

    if not np.isclose(forward_log_prob, backward_log_prob, rtol=TOLERANCE, atol=0):
        raise ValueError("forward and backward log P(x) don't match")

    posterior_matrix = np.exp(forward_matrix + backward_matrix - forward_log_prob)

    if not np.allclose(posterior_matrix.sum(axis=0), 1, rtol=TOLERANCE, atol=0):
        raise ValueError("a posterior column doesn't sum to 1")

    return {"posterior": posterior_matrix, "forward": forward_matrix, "backward": backward_matrix, "log_prob": forward_log_prob}


In [3]:
# Example HMM from Project 09 instructions

obs = "ATGCAA"

init_probs = {
    "E": 0.6,
    "I": 0.4
}

trans_probs = {
    "E": {"E": 0.8, "I": 0.2},
    "I": {"E": 0.3, "I": 0.7}
}

emit_probs = {
    "E": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1}
}

hmm = HMM(init_probs, trans_probs, emit_probs)